In [23]:
import pandas as pd

## Step 0: Load & Combine Summaries

In [24]:
import glob
import os

summary_files = glob.glob('/home/vscode/workspace/summaries/experiment-*/summaries.csv')
dfs = []
for f in summary_files:
    df = pd.read_csv(f)
    df['source_file'] = os.path.basename(os.path.dirname(f))
    dfs.append(df)

combined_df = pd.concat(dfs, ignore_index=True)
print(f"Loaded {len(summary_files)} files, {len(combined_df)} total rows")

Loaded 1 files, 100 total rows


In [25]:
# clean empty and nan rows 
combined_df = combined_df.dropna(subset=['semantic_summary'])

## Step 1: Encode Summaries

In [ ]:
from langchain_core.embeddings import Embeddings
from pydantic import Field
import requests

class LMStudioEmbeddings(Embeddings):
    base_url: str = 'http://host.docker.internal:1234/v1'
    model: str = 'nomic-embed-text'
    
    def _embed(self, texts: list[str]) -> list[list[float]]:
        response = requests.post(
            f'{self.base_url}/embeddings',
            json={'input': texts, 'model': self.model}
        )
        response.raise_for_status()
        return [item['embedding'] for item in response.json()['data']]
    
    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return self._embed(texts)
    
    def embed_query(self, text: str) -> list[float]:
        return self._embed([text])[0]

embeddings = LMStudioEmbeddings(
)

In [29]:
# Test embedding
test_result = embeddings.embed_query('this is a test')
print(f"Embedding dimension: {len(test_result)}")

TypeError: Object of type FieldInfo is not JSON serializable

In [ ]:
# Apply embeddings to all summaries
combined_df['embedding'] = combined_df['semantic_summary'].apply(
    lambda x: embeddings.embed_query(x)
)

## Step 2: Calculate Distances

In [ ]:
from sklearn.metrics.pairwise import cosine_distances
from itertools import combinations

print("Unique commits per source:")
print(combined_df.groupby('source_file')['commit hash'].nunique())

distance_records = []
for commit_hash, group in combined_df.groupby('commit hash'):
    if len(group) < 2:
        continue
    embedding_list = list(group['embedding'])
    sources = list(group['source_file'])
    for (i, j) in combinations(range(len(embedding_list)), 2):
        dist = cosine_distances([embedding_list[i]], [embedding_list[j]])[0][0]
        distance_records.append({
            'commit_hash': commit_hash,
            'source_1': sources[i],
            'source_2': sources[j],
            'distance': dist
        })

distances_df = pd.DataFrame(distance_records)
print(f"Calculated {len(distances_df)} pairwise distances")

## Step 3: Rank by Distance

In [ ]:
if len(distances_df) == 0:
    print("No pairwise distances found. Need multiple experiment folders with the same commits.")
else:
    global_mean = distances_df['distance'].mean()
    global_std = distances_df['distance'].std()
    distances_df['distance_zscore'] = (distances_df['distance'] - global_mean) / global_std

    ranked_df = distances_df.sort_values('distance', ascending=False)
    print(f"Global mean: {global_mean:.4f}, Global std: {global_std:.4f}")
    ranked_df.head(10)